# Lab 05: Parameter-Efficient Fine-Tuning (PEFT / LoRA)

In this lab, you will explore Parameter-Efficient Fine-Tuning (PEFT):
1. Inspect model configuration weights stored in `/mnt/storage/models`.
2. Inject low-rank trainable adapter matrices (LoRA) into linear projection layers using HuggingFace `peft`.
3. Execute fine-tuning steps (auto-selecting `cuda` if available or `cpu`).
4. Stream adapter training loss metrics and configuration artifacts to central MLflow.

In [ ]:
import os
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
import mlflow

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"⚡ Compute Device: {device}")
if device == "cuda":
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")

In [ ]:
# Define base model architecture (or wrap local transformer weights)
class LanguageHeadModel(nn.Module):
    def __init__(self, vocab_size=1000, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, input_ids):
        x = self.embedding(input_ids)
        q = self.q_proj(x)
        v = self.v_proj(x)
        return self.lm_head(q + v)

base_model = LanguageHeadModel().to(device)
total_params = sum(p.numel() for p in base_model.parameters())
print(f"🧠 Base Model Total Parameters: {total_params:,}")

In [ ]:
# Configure LoRA Adapter targeting projection layers
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

peft_model = get_peft_model(base_model, lora_config)
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"🎯 Trainable LoRA Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}% of total)")

In [ ]:
# Execute Fine-Tuning Step & Log Telemetry to MLflow
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow.services.svc.cluster.local:5000")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("starter_lab_05_lora")

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

dummy_tokens = torch.randint(0, 1000, (4, 16)).to(device)
dummy_targets = torch.randint(0, 1000, (4, 16)).to(device)

with mlflow.start_run(run_name="lora_adapter_finetune") as run:
    mlflow.log_params({"lora_r": 8, "lora_alpha": 16, "device": device, "lr": 1e-3})
    
    peft_model.train()
    for step in range(1, 4):
        optimizer.zero_grad()
        logits = peft_model(dummy_tokens)
        loss = criterion(logits.view(-1, 1000), dummy_targets.view(-1))
        loss.backward()
        optimizer.step()
        
        mlflow.log_metric("train_loss", loss.item(), step=step)
        print(f"  Step {step}/3 - Loss: {loss.item():.4f}")
    
    adapter_dir = "/mnt/storage/scratch/lora_adapter_output"
    peft_model.save_pretrained(adapter_dir)
    mlflow.log_artifacts(adapter_dir, artifact_path="lora_weights")
    print(f"💾 Saved and logged LoRA adapter weights from {adapter_dir}")

print("🎉 Lab 05 Completed Successfully!")